# Transfers: which arm §5.3.2 takes, and whether the night can carry it

§5.3.2 gives the rebalancing rule in one sentence, and there is more in the
sentence than there looks: a gate, then a cost comparison, then a third outcome
for when neither arm is open. This notebook takes one over-full depot and moves
one input at a time until each of the three arms comes out.

It then hands the decision to the night, because §4.2 deciding a transfer and
§5.3 being able to fly it are two different questions.

**Every number here is provisional.** `assumptions.BIKE_RELOCATION_MIN` is
invented — §4.2 says only that "relocation between distant facilities has a time
cost" — and the exchange rate between a van-minute and a bike-minute is §12 Q15,
still unanswered. `docs/assumptions.md` labels both.

In [ ]:
from datetime import date, datetime, time

from ddn import assumptions, linehaul
from ddn.allocation import EFFECTIVE_PER_BIKE, Arm, allocate, decide, place
from ddn.model import TransferReason, TransferRequest
from ddn.simulation.day import RETRIABLE_DECLINES

HOUR = 3600
TODAY = date(2026, 9, 16)
# §3.1 and §5.6 give clock times with no zone; the operation runs in one place,
# so datetimes here are naive and local, as they are throughout the modules.
RELEASE = datetime.combine(date(2026, 9, 17), time(7))
RAISED = datetime.combine(TODAY, time(18))

## One depot over, two with slack

§4.2 splits a fixed fleet across the depots by today's demand share. Tomorrow's
projection is what §5.3.2 then argues about — and the argument only starts
where a depot is projected past what its own bikes can serve.

In [ ]:
today = {"D1": 200, "D2": 200, "D3": 100}
fleet = [f"BIKE-{n:02d}" for n in range(1, 21)]
allocation = place(allocate(today, len(fleet)), fleet)

bikes = {f: len(ids) for f, ids in allocation.by_facility().items()}
capacity = {f: n * EFFECTIVE_PER_BIKE for f, n in bikes.items()}
projected = {"D1": 240, "D2": 160, "D3": 50}

print(f"{'':<6}{'bikes':>7}{'capacity':>10}{'tomorrow':>10}{'over/room':>11}")
for facility in sorted(capacity):
    gap = projected[facility] - capacity[facility]
    print(f"{facility:<6}{bikes[facility]:>7}{capacity[facility]:>10}"
          f"{projected[facility]:>10}{gap:>+11}")

## The offer (§5.3)

`linehaul.rebalancing` moves nothing. It returns `Proposal`, not
`TransferRequest`, because "§4.2 owns the fleet, and a planner that rebalanced on
its own authority would be making an allocation decision from inside §5.3".

`reaches` is a precondition, not an aspiration: D3 has the most room tonight and
no van circuit goes there, so the offer never mentions it.

In [ ]:
pool = [{"package_id": f"P{n:02d}", "priority": 1000 + n} for n in range(240)]
offers = linehaul.rebalancing(
    projected, capacity, pools={"D1": pool},
    reaches=lambda source, destination: (source, destination) == ("D1", "D2"))

offer, = offers
print(f"{offer.from_facility_id} -> {offer.to_facility_id}: "
      f"{len(offer.package_ids)} envelopes over leg {offer.leg}")
print("  offered:", ", ".join(offer.package_ids[:3]), "...", offer.package_ids[-1])

# §8.1 ranks what is delivered and what is moved is the mirror of it, so the
# envelopes offered are the depot's *lowest* scoring. Asserted rather than
# described: the sentence above would go on reading well if the sort reversed.
assert offer.package_ids == tuple(f"P{n:02d}" for n in range(40))

## A gate, then a comparison

> "the choice between moving envelopes and moving motorbikes is a cost comparison
> made by the allocation step: transfer if a van leg between the two depots
> already exists or can be added within van-hours, and the moved envelopes arrive
> before the receiving depot's morning release; otherwise reallocate motorbikes
> or leave the envelopes unassigned by priority." — §5.3.2

The `if` is feasibility — a leg, van-hours, an arrival before the release — and
"cost comparison" decides between the arms that clear it. The trailing clause is
a third arm rather than an error: a depot no leg can drain and no spare bike can
serve rolls its lowest-priority envelopes, which is §8.1 working.

The two arms spend different resources and the document gives no exchange rate
(§12 Q15). **The reading taken here is one minute against one minute**, because
it is the only one that needs no number the document withholds.

In [ ]:
candidates = {pid: TransferRequest(
    transfer_id=f"T-{pid}", package_id=pid, from_facility_id="D1",
    to_facility_id="D2", reason=TransferReason.REBALANCING, created_at=RAISED,
    deadline=RELEASE, priority=1000.0) for pid in offer.package_ids}


def arm_for(**changed):
    """§5.3.2 with one input moved; everything else is tonight as planned."""
    tonight = {
        "candidates": candidates, "van_hours": 2.0, "projected": projected,
        "capacity": capacity, "added_leg_minutes": {("D1", "D2"): 40.0},
        "arrivals": {("D1", "D2"): datetime.combine(date(2026, 9, 17), time(5))}}
    return decide(offers, allocation, **(tonight | changed))


past_release = datetime.combine(date(2026, 9, 17), time(8))
cases = {
    "nothing — the leg costs 40 van-minutes": {},
    "the leg costs 120 van-minutes": {"added_leg_minutes": {("D1", "D2"): 120.0}},
    "no leg can be added at all": {"added_leg_minutes": {}},
    "the van lands at 08:00, past D2's release": {
        "arrivals": {("D1", "D2"): past_release}},
    "half an hour of van-hours is left": {"van_hours": 0.5},
    "no leg, and the neighbours are full too": {
        "added_leg_minutes": {}, "projected": {"D1": 240, "D2": 200, "D3": 100}},
}

print(f"{'what changed':<44}{'arm':<12}{'carried':>8}{'rolled':>8}   bikes after")
arms = []
for label, changed in cases.items():
    decision = arm_for(**changed)
    arms.append(decision.by_depot["D1"])
    after = {f: len(ids) for f, ids in decision.allocation.by_facility().items()}
    print(f"{label:<44}{decision.by_depot['D1']:<12}{len(decision.transfers):>8}"
          f"{len(decision.unassigned.get('D1', ())):>8}   {after}")

# The paragraph under here reads the arms off in this order.
assert arms == [Arm.TRANSFER, Arm.REALLOCATE, Arm.REALLOCATE, Arm.REALLOCATE,
                Arm.REALLOCATE, Arm.UNASSIGNED]
assert arm_for().allocation is allocation, (
    "unchanged and *identical*, so a caller can tell 'decided nothing' from "
    "'decided to keep'")

Six runs, one input each, all three arms.

Only the **second** row is the comparison. Two bikes at `BIKE_RELOCATION_MIN`
(45 minutes each) is 90 minutes of equivalent cost, so a 40-minute leg wins and a
120-minute leg loses. That boundary is the line §12 Q15 will move.

Rows three to five are the gate rather than the comparison — no leg, an arrival
after D2's 07:00 release, no van-hours left. Each shuts it for a different
reason and all three fall through to the same arm, because the reallocation arm
was open the whole time.

The last row shuts both. D2 and D3 are projected to their own capacity, so
neither has a whole bike of room to spare, and forty envelopes roll by priority.

§7.2 lists "rebalancing by transfer when reallocating motorbikes would have been
cheaper, or vice versa" as a soft constraint. That is the measurement that will
say whether par was the wrong reading — this notebook cannot.

## §4.2 deciding is not §5.3 flying

A `TransferRequest` is a commitment, and the night still has to have a van going
that way. `linehaul.plan` boards them against the same 500 kg and the same
morning releases as everything else, and §5.3.2's transfer loads are "picked up
at earlier depots" — so the circuit gains a stop it would not otherwise make.

In [ ]:
def depot(name, *, transit_min, release_h=7):
    return {"id": name, "route_release_time": release_h * HOUR,
            "transit_from_hub_min": transit_min}


def envelope(n, *, grams=200, priority=1.0):
    return {"package_id": f"D1-{n}", "facility_id": "D1", "weight_g": grams,
            "expected_ready_at": 16 * HOUR, "priority": priority}


def transit(_origin, _destination):
    """§3.1 supplies hub transit only, so inter-depot seconds are the caller's."""
    return 40 * 60


DEPOTS = [depot("D1", transit_min=30), depot("D2", transit_min=45)]
VAN = [{"vehicle_id": "VAN-01"}]
UNLOAD = assumptions.FACILITY_UNLOAD_MIN * 60

night = linehaul.plan(DEPOTS, [envelope(n) for n in range(3)], VAN,
                      unload_seconds=UNLOAD, transfers=arm_for().transfers,
                      transit=transit)
trip, = night.trips
for leg in trip.legs:
    print(f"  {leg.from_facility:<4}-> {leg.to_facility:<4}{leg.weight_g:>8} g")
print(f"\n{len(trip.transfer_ids)} carried, {len(night.declined)} declined")

assert [(leg.from_facility, leg.to_facility) for leg in trip.legs] == [
    ("HUB", "D1"), ("D1", "D2"), ("D2", "HUB")], "§5.3.2: the circuit is the hub's"
assert trip.legs[1].weight_g == 40 * 200, "only the transfer, after D1 unloads"

### Three refusals, and which of them come back

§9.2 asks for "transfers not carried, with reason". There are three reasons and
they are not the same kind of no.

In [ ]:
def request(tid, *, frm="D1", to="D2", grams=200):
    return TransferRequest(tid, f"PKG-{tid}", frm, to,
                           TransferReason.REBALANCING, RAISED, RELEASE,
                           weight_g=grams, priority=100.0)


# A van still on §5.1's pickups at 06:00 is available for line-haul and useless
# for it: `linehaul_release_at` past 30 h leaves no room to reach D2 by seven.
off_pickups_late = [{"vehicle_id": "VAN-01", "role": "pickup",
                     "linehaul_release_at": 30 * HOUR}]
refusals = (
    ("from a depot no circuit visits", [request("T-away", frm="D5", to="D6")],
     [envelope(n) for n in range(3)], VAN),
    ("the van is off pickups at 06:00", [request("T-late")],
     [envelope(n) for n in range(3)], off_pickups_late),
    ("500 kg of higher-scoring envelopes", [request("T-heavy", grams=1000)],
     [envelope(0, grams=500_000, priority=900.0)], VAN),
)

reasons = []
for label, wanted, envelopes, vans in refusals:
    refused = linehaul.plan(DEPOTS, envelopes, vans, unload_seconds=UNLOAD,
                            transfers=wanted, transit=transit)
    declined, = refused.declined
    reasons.append(declined.reason)
    again = "retried tomorrow" if declined.reason in RETRIABLE_DECLINES else "final"
    print(f"{label:<36}{declined.reason:<52}{again}")

assert reasons == [linehaul.NO_VAN_LEG, linehaul.MISSES_DEADLINE,
                   linehaul.OVER_CAPACITY]

Two of the three are about tonight; one is about the envelope.

`simulation.day` retries `NO_VAN_LEG` and `OVER_CAPACITY` on the next night,
because another night has another fleet. `MISSES_DEADLINE` is final: §9.1 fixes a
transfer's deadline when it is raised, and an envelope that cannot make the
receiving depot's release tonight does not make it by waiting.

Which is the shape of the arms above as well. §5.3.2's last clause is not a
failure path either — it is the document saying what happens when both arms are
shut, and a `Decision` that could not express it would be dropping a clause of
the rule.